# 03 - Agentic RAG Implementation

## 🎯 Learning Objectives

By the end of this notebook, you will:
- Understand Agentic RAG architecture
- Build an agent that decides when to retrieve
- Create retrieval tools for agents
- Compare agentic vs 2-step RAG
- Handle multi-step reasoning with retrieval

## 🤖 What is Agentic RAG?

**Agentic RAG** combines retrieval with agent-based reasoning. Instead of always retrieving before answering, an LLM-powered agent reasons step-by-step and decides **when** and **how** to retrieve information.

```plain
User Question → Agent (LLM) → Need info? → Search Tool → Enough? → Generate Answer
                     ↑                           ↓
                     └───────────────────────────┘
```

### Characteristics:

| Feature | Description |
|---------|-------------|
| **Control** | ❌ Low - agent decides the flow |
| **Flexibility** | ✅ High - can retrieve 0 to N times |
| **Latency** | ⏳ Variable - depends on reasoning steps |
| **Use Cases** | Research assistants, complex Q&A, multi-tool workflows |

### When to use Agentic RAG:
- Complex questions requiring multi-step reasoning
- Uncertain if retrieval is needed
- Access to multiple knowledge sources
- Need for decision-making about what to retrieve

## 🔧 Setup

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv("../.env")

print("✅ Environment loaded")

✅ Environment loaded


In [2]:
from langchain_dev_utils.chat_models import register_model_provider, load_chat_model
from langchain_dev_utils.embeddings import register_embeddings_provider, load_embeddings

# SiliconFlow configuration
SILICONFLOW_BASE_URL = os.getenv("SILICONFLOW_BASE_URL", "https://api.siliconflow.cn/v1")

# Register providers
register_model_provider(
    provider_name="siliconflow",
    chat_model="openai-compatible",
    base_url=SILICONFLOW_BASE_URL,
)

register_embeddings_provider(
    provider_name="siliconflow",  # Fixed: use provider_name instead of provider
    embeddings_model="openai-compatible",
    base_url=SILICONFLOW_BASE_URL,
)

# Load models
CHAT_MODEL_NAME = os.getenv("SILICONFLOW_CHAT_MODEL", "Qwen/Qwen2.5-7B-Instruct")
EMBEDDING_MODEL_NAME = os.getenv("SILICONFLOW_EMBEDDING_MODEL", "BAAI/bge-m3")

chat_model = load_chat_model(f"siliconflow:{CHAT_MODEL_NAME}")
embeddings = load_embeddings(f"siliconflow:{EMBEDDING_MODEL_NAME}")

print(f"✅ Chat Model: {CHAT_MODEL_NAME}")
print(f"✅ Embedding Model: {EMBEDDING_MODEL_NAME}")

✅ Chat Model: Qwen/Qwen2.5-7B-Instruct
✅ Embedding Model: BAAI/bge-m3


In [3]:
from langchain_oceanbase.vectorstores import OceanbaseVectorStore

# OceanBase connection
connection_args = {
    "host": os.getenv("OCEANBASE_HOST", "127.0.0.1"),
    "port": int(os.getenv("OCEANBASE_PORT", "2881")),
    "user": os.getenv("OCEANBASE_USER", "root@test"),
    "password": os.getenv("OCEANBASE_PASSWORD", ""),
    "db_name": os.getenv("OCEANBASE_DB", "test"),
}

# Load existing vector store
vector_store = OceanbaseVectorStore(
    embedding_function=embeddings,
    table_name="langchain_knowledge_base",
    connection_args=connection_args,
    vidx_metric_type="cosine",
    drop_old=False,
)

print("✅ Connected to knowledge base")

✅ Connected to knowledge base


## 🔨 Step 1: Create Retrieval Tools

The key to Agentic RAG is giving the agent **tools** to retrieve information.

In [9]:
from langchain.tools import tool
from typing import List

@tool
def search_knowledge_base(query: str, top_k: int = 3) -> str:
    """Search Nike's 10-K annual report for relevant business information.
    
    Use this tool when you need to find information about:
    - Nike's financial performance (revenues, profits, earnings)
    - Business segments and product categories
    - Geographic operations and markets
    - Risk factors and challenges
    - Strategic initiatives and growth plans
    - Competitive position and market trends
    
    Args:
        query: The search query describing what you want to find about Nike
        top_k: Number of relevant documents to retrieve (default: 3)
    
    Returns:
        A formatted string with the retrieved documents and their page numbers
    """
    # Create retriever with dynamic k parameter
    retriever = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": top_k}
    )
    
    # Perform retrieval
    results = retriever.invoke(query)
    
    if not results:
        return "No relevant information found in Nike's 10-K report."
    
    # Format results
    formatted_results = []
    for i, doc in enumerate(results, 1):
        page = doc.metadata.get('page', 'Unknown')
        formatted_results.append(
            f"Document {i} [Page {page}]:\n{doc.page_content}\n"
        )
    
    return "\n".join(formatted_results)

print("✅ Created search_knowledge_base tool for Nike 10-K data")

✅ Created search_knowledge_base tool for Nike 10-K data


In [10]:
# Test the tool directly
test_result = search_knowledge_base.invoke({"query": "Nike revenue 2023", "top_k": 2})
print("🔍 Tool Test:")
print(test_result[:500] + "..." if len(test_result) > 500 else test_result)

🔍 Tool Test:
Document 1 [Page 30]:
speed and responsiveness as we serve consumers globally.
FINANCIAL HIGHLIGHTS
• In fiscal 2023, NIKE, Inc. achieved record Revenues of $51.2 billion, which increased 10% and 16% on a reported and currency-neutral basis, respectively
• NIKE Direct revenues grew 14% from $18.7 billion in fiscal 2022 to $21.3 billion in fiscal 2023, and represented approximately 44% of total NIKE Brand revenues for
fiscal 2023
• Gross margin for the fiscal year decreased 250 basis points to 43...


## 🤖 Step 2: Create the Agent

Now let's create an agent using LangChain v1's `create_agent` - the new standard for building agents.

In [11]:
from langchain.agents import create_agent

# Create tools list
tools = [search_knowledge_base]

# Create agent using LangChain v1 create_agent
# This is the new standard way to build agents in LangChain v1
agent = create_agent(
    model=chat_model,
    tools=tools,
    system_prompt="You are a helpful AI assistant with access to Nike's 10-K annual report. Use the search_knowledge_base tool when you need information about Nike's business, financials, segments, risks, or strategy. For simple questions that don't require Nike data, answer directly."
)

print("✅ Agentic RAG system created for Nike 10-K analysis")
print(f"🔧 Available tools: {[tool.name for tool in tools]}")
print("\n💡 Using LangChain v1 create_agent (replaces deprecated AgentExecutor)")

✅ Agentic RAG system created for Nike 10-K analysis
🔧 Available tools: ['search_knowledge_base']

💡 Using LangChain v1 create_agent (replaces deprecated AgentExecutor)


## 🎯 Step 3: Test Agentic RAG

### Test 1: Question requiring retrieval

In [12]:
question1 = "What were Nike's total revenues and operating income in fiscal 2023?"

print("="*80)
print(f"❓ Question: {question1}")
print("="*80)

# Invoke agent with LangChain v1 message format
result1 = agent.invoke({"messages": [{"role": "user", "content": question1}]})

print("\n" + "="*80)
print("✅ Final Answer:")
print("="*80)
# Extract the final message content
print(result1["messages"][-1].content)

❓ Question: What were Nike's total revenues and operating income in fiscal 2023?

✅ Final Answer:
In fiscal 2023, Nike's total revenues were $51,217 million and the operating income was $6,201 million. These figures represent a 10% increase in revenues and a 7% decrease in operating income when compared to fiscal 2022.


#### 📊 View Agent Execution Trace

You can view the detailed execution trace in LangSmith to see how the agent:
- Decided whether to use the search tool
- Made multiple tool calls if needed
- Reasoned step-by-step to answer the question

**Example trace**: [View in LangSmith](https://smith.langchain.com/public/1c520301-b171-4a1c-9f6d-90d308f7352e/r)

The trace shows:
- 🔍 **Tool calls**: When and what the agent searched for
- 💭 **Reasoning steps**: The agent's thought process
- ⏱️ **Latency breakdown**: Time spent on each step
- 📝 **Input/Output**: Full message history

💡 **Tip**: Enable LangSmith tracing in your `.env` file to capture your own traces!

### Test 2: Simple question (may not need retrieval)

In [13]:
question2 = "What is 2 + 2?"

print("="*80)
print(f"❓ Question: {question2}")
print("="*80)

result2 = agent.invoke({"messages": [{"role": "user", "content": question2}]})

print("\n" + "="*80)
print("✅ Final Answer:")
print("="*80)
print(result2["messages"][-1].content)
print("\n💡 Notice: Agent decided NOT to use retrieval for this simple question!")

❓ Question: What is 2 + 2?

✅ Final Answer:
2 + 2 equals 4. If you have any other questions, feel free to ask.

💡 Notice: Agent decided NOT to use retrieval for this simple question!


#### 📊 Compare Trace Behavior

Notice how the agent's trace differs for this simple question:
- No tool calls made
- Direct answer without retrieval
- Lower latency

**Example trace**: [View in LangSmith](https://smith.langchain.com/public/85ea4840-088a-4aa7-88d2-3c5a6b9ccfc7/r)

This demonstrates the agent's intelligence in deciding when retrieval is unnecessary. Compare this trace with Test 1 to see the difference!

### Test 3: Multi-step question

In [15]:
question3 = """First, tell me about Nike's digital commerce and e-commerce strategy. 
Then, explain their sustainability initiatives and environmental commitments. 
Finally, describe their innovation and product development approach."""

print("="*80)
print(f"❓ Question: {question3}")
print("="*80)

result3 = agent.invoke({"messages": [{"role": "user", "content": question3}]})

print("\n" + "="*80)
print("✅ Final Answer:")
print("="*80)
print(result3["messages"][-1].content)
print("\n💡 Notice: This multi-part question should trigger multiple searches for different topics!")

❓ Question: First, tell me about Nike's digital commerce and e-commerce strategy. 
Then, explain their sustainability initiatives and environmental commitments. 
Finally, describe their innovation and product development approach.

✅ Final Answer:
### Digital Commerce and E-Commerce Strategy

Nike's strategy emphasis on digital commerce and e-commerce is rooted in providing attractive, effective, reliable, secure, and user-friendly platforms that cater to the changing expectations of online shoppers. Key aspects include:

1. **Interactive Digital Platforms:** Nike uses social media and proprietary mobile applications to interact with consumers and enhance their shopping experience. This interaction helps in building strong brand relationships and drives engagement.
2. **Consumer Direct Acceleration (CDA):** Under this strategy, Nike aims to create a future marketplace with premium, consistent, and seamless consumer experiences. The focus is on digital and owned stores, as well as selec

#### 📊 Multi-Step Reasoning Trace

For this multi-part question, check the trace to see:
- **Multiple tool calls**: Agent searches separately for:
  1. "digital commerce e-commerce strategy"
  2. "sustainability environmental commitments"
  3. "innovation product development"
- **Sequential reasoning**: Agent builds answer in logical order
- **Synthesis**: Combines multiple search results into structured response

**Example trace**: [View in LangSmith](https://smith.langchain.com/public/a2657483-6362-4602-9322-dc9f1e24dae6/r)

This demonstrates how agentic RAG handles complex, multi-faceted questions by breaking them down into retrievable sub-queries.

## 📊 Step 4: Compare Agentic vs 2-Step RAG

Let's compare the two approaches side by side.

In [ ]:
# Reload 2-Step RAG from notebook 02
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Create retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 2-Step RAG chain for Nike data
two_step_rag = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | ChatPromptTemplate.from_template(
        """Answer based on Nike's 10-K report context:

{context}

Question: {question}

Answer:"""
    )
    | chat_model
    | StrOutputParser()
)

print("✅ 2-Step RAG loaded for comparison")

In [22]:
import time

test_questions = [
    "What are Nike's key risk factors?",
    "Hello, how are you?",  # Doesn't need retrieval
    "Compare Nike's revenue across different geographic regions",
]

print("📊 COMPARISON: Agentic RAG vs 2-Step RAG")
print("="*80)

for q in test_questions:
    print(f"\n{'─'*80}")
    print(f"❓ Question: {q}")
    print(f"{'─'*80}")
    
    # 2-Step RAG (always retrieves)
    start = time.time()
    two_step_answer = two_step_rag.invoke(q)
    two_step_time = time.time() - start
    
    print(f"\n🔵 2-Step RAG (time: {two_step_time:.2f}s):")
    print(f"   Always retrieves → {two_step_answer[:100]}...")
    
    # Agentic RAG (decides whether to retrieve)
    start = time.time()
    agent_result = agent.invoke({"messages": [{"role": "user", "content": q}]})
    agent_time = time.time() - start
    
    # Extract answer from messages
    agent_answer = agent_result["messages"][-1].content
    
    print(f"\n🟢 Agentic RAG (time: {agent_time:.2f}s):")
    print(f"   Decides when to retrieve → {agent_answer[:100]}...")

print(f"\n{'='*80}")
print("\n💡 Key Observations:")
print("   • 2-Step RAG: Fast and predictable, but always retrieves from Nike 10-K")
print("   • Agentic RAG: Flexible but variable latency, retrieves only when needed")

📊 COMPARISON: Agentic RAG vs 2-Step RAG

────────────────────────────────────────────────────────────────────────────────
❓ Question: What are Nike's key risk factors?
────────────────────────────────────────────────────────────────────────────────

🔵 2-Step RAG (time: 7.57s):
   Always retrieves → Based on Nike's 2023 Form 10-K report, some of the key risk factors for Nike include:

- **Market Co...

🟢 Agentic RAG (time: 7.77s):
   Decides when to retrieve → Here are some key risk factors identified in Nike's 10-K annual report:

1. **Global Health Crises a...

────────────────────────────────────────────────────────────────────────────────
❓ Question: Hello, how are you?
────────────────────────────────────────────────────────────────────────────────

🔵 2-Step RAG (time: 1.52s):
   Always retrieves → Hello! I'm here and ready to help. How can I assist you based on the context provided from Nike's 10...

🟢 Agentic RAG (time: 0.95s):
   Decides when to retrieve → Hello! I'm just a comp

### 📊 Exploring Traces with LangSmith

Now that you've run both RAG approaches, it's time to dive deep into their execution patterns!

**🔍 How to View Your Traces:**

1. **Enable LangSmith** in your `.env` file:
   ```bash
   LANGCHAIN_TRACING_V2=true
   LANGCHAIN_API_KEY=your-api-key
   LANGCHAIN_PROJECT=ep2-langchain-retrieval
   ```

2. **Re-run the comparison above** - Your traces will be automatically captured

3. **Visit LangSmith** at https://smith.langchain.com and navigate to your project

**💡 What to Look For in Traces:**

**2-Step RAG patterns**:
- ✅ Always shows retrieval step first
- ✅ Predictable: Retrieve → Generate → Done
- ✅ Consistent latency (1 retrieval + 1 LLM call)
- ⚠️ Even for "Hello, how are you?" it retrieves Nike data

**Agentic RAG patterns**:
- 🎯 Variable execution paths based on question
- 🎯 Zero tool calls for simple questions ("Hello", "2+2")
- 🎯 Single tool call for straightforward Nike questions
- 🎯 Multiple tool calls for complex, multi-faceted questions
- 🎯 Agent reasoning visible in trace steps

**🎓 Learning Exercise:**

Compare the traces for these three questions and note:
1. "Hello, how are you?" - How many tool calls? Why?
2. "What are Nike's key risk factors?" - Single or multiple retrievals?
3. "Compare Nike's revenue across regions" - How does the agent break this down?

Understanding traces is crucial for debugging and optimizing agent behavior in production!

## 🎉 Summary

You've successfully implemented **Agentic RAG** with Nike's 10-K annual report using **LangChain v1**!

### What we built:

- ✅ **Retrieval Tool**: Created a tool for searching Nike's 10-K report using `@tool` decorator
- ✅ **LangChain v1 Agent**: Built an agent with `create_agent()` - the new standard in LangChain v1
- ✅ **Multi-Step Reasoning**: Agent can search multiple times for complex Nike questions
- ✅ **Intelligent Decision-Making**: Agent skips retrieval when not needed (e.g., greetings, math)
- ✅ **Comparison**: Analyzed Agentic vs 2-Step RAG with Nike business questions

### LangChain v1 Agent Pattern:

```python
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def my_tool(query: str) -> str:
    """Tool description"""
    return result

agent = create_agent(
    model=chat_model,
    tools=[my_tool],
    system_prompt="Agent instructions"
)

# Invoke with messages
result = agent.invoke({"messages": [{"role": "user", "content": "question"}]})
```

### Key Advantages of v1 `create_agent`:

- ✅ **Simpler API**: No need for AgentExecutor or custom prompt templates
- ✅ **Built on LangGraph**: Durable execution, streaming, human-in-the-loop support
- ✅ **Production-ready**: Designed for real-world agent applications
- ✅ **Standard interface**: Unified message format across all LangChain agents

### Agentic RAG Characteristics:

| Aspect | Description |
|--------|-------------|
| **Flexibility** | Agent decides if, when, and how many times to retrieve |
| **Intelligence** | Can skip retrieval for simple questions |
| **Multi-step** | Can retrieve multiple times for complex questions |
| **Latency** | Variable - depends on agent's reasoning path |
| **Transparency** | Built-in logging and tracing with LangSmith |

### When to use Agentic RAG:

✅ **Good for**:
- Financial research requiring multiple data points (like segment analysis)
- Complex questions needing iterative information gathering
- Scenarios where retrieval isn't always needed (mixed question types)
- Multi-source knowledge integration

❌ **Not ideal for**:
- Simple, single-fact queries (use 2-Step RAG)
- When latency must be consistent and predictable
- Production systems requiring strict cost controls
- High-volume, routine queries

### Comparison: 2-Step vs Agentic RAG

| Feature | 2-Step RAG | Agentic RAG |
|---------|------------|-------------|
| **Control** | ✅ High | ❌ Low |
| **Flexibility** | ❌ Low | ✅ High |
| **Latency** | ⚡ Predictable | ⏳ Variable |
| **Complexity** | 🟢 Simple | 🟡 Moderate |
| **Cost** | 💰 Fixed (1 retrieval + 1 LLM) | 💰💰 Variable (N retrievals + M LLM calls) |

### Real-world Use Cases:

**2-Step RAG** is better for:
- "What was Nike's revenue in 2023?" (single fact lookup)
- FAQ-style queries with consistent format
- High-volume customer support queries

**Agentic RAG** is better for:
- "Compare Nike's performance across segments and identify growth opportunities" (multi-step analysis)
- Research tasks requiring synthesis of multiple data points
- Exploratory analysis where the information need evolves

### Next Steps:

In **Notebook 05**, we'll implement **Hybrid Search**:
- Understand vector, sparse, and full-text search modalities
- Configure OceanBase for hybrid search
- Generate sparse vectors for keyword matching
- Implement full-text search with phrase matching
- Combine all three with weighted fusion

## 💡 Additional Resources

- [LangChain Agents Documentation](https://python.langchain.com/docs/modules/agents/)
- [Tool Calling Guide](https://python.langchain.com/docs/how_to/tool_calling/)
- [ReAct Agent Pattern](https://arxiv.org/abs/2210.03629)
- [Agent Architectures](https://python.langchain.com/docs/concepts/agents/)